In [ ]:
import pandas as pd

files = {
    'Ooredoo': '../data/raw/posts_dataOOFB0.csv',
    'Orange': '../data/raw/posts_dataOrangeFB0.csv',
    'TT': '../data/raw/posts_dataTTFB0.csv'
}

dfs = []
for name, path in files.items():
    tmp = pd.read_csv(path, encoding='latin-1', on_bad_lines='skip', engine='python')
    tmp['entreprise'] = name
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)
print(df.shape)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
reaction_cols = ["likes", "wow", "sad", "haha", "angry", "love", "thankful"]
for col in reaction_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['reactions'] = df[reaction_cols].sum(axis=1, min_count=1)

In [ ]:
print(df['reactions'] )

In [ ]:
# Features simples à partir du message, avant suppression
df['message_length'] = df['message'].fillna('').apply(len)
df['has_hashtag'] = df['message'].fillna('').str.contains('#').astype(int)
df['has_link'] = df['message'].fillna('').str.contains('http|www\.').astype(int)
df['has_emoji'] = df['message'].fillna('').str.contains(
    r'[\U0001F300-\U0001FAFF\U00002600-\U000027BF]', regex=True
).astype(int)

# Suppression des colonnes (reach_organic/reach_paid conservées pour Orange)
df.drop(columns=['created_at','updated_at','is_deleted','is_real','url', 'message','story',
'picture','profile_cover','local_media','data','views_organic','views_paid','likes','wow',
'sad', 'haha', 'angry', 'love', 'thankful', 'none',
'comments_ad','views_ad','reach_ad','likes_ad','saved_ad','shares_ad','reactions_new',
'score_new','tag','type',"id","saved","commentaires"], axis=1, inplace=True)

In [ ]:
df.info()

In [ ]:
# Suppression des colonnes avec plus de 90% de valeurs manquantes
missing_pct = df.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 90].index.tolist()

print(f"Colonnes supprimées (>90% manquant): {cols_to_drop}")
df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
from datetime import datetime
df = df.rename(columns={"creation_time": "timestamp"})
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%d/%m/%Y %H:%M", errors='coerce')

In [ ]:
print(f"Timestamps invalides: {df['timestamp'].isna().sum()} / {len(df)}")

In [ ]:
df = df.dropna(subset=['timestamp']).reset_index(drop=True)

In [ ]:
df = df.rename(columns={
    "status_type": "content_categorie",
    "social_type": "platform"
})

# page_id n'est plus utile, on le supprime puisqu'on a "entreprise"
if "competitor_id" in df.columns:
    df.drop(columns=["competitor_id"], inplace=True)

In [ ]:
df.info()

In [ ]:
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Lundi
df['day_name'] = df['timestamp'].dt.day_name()
df['month'] = df['timestamp'].dt.month
df['year'] = df['timestamp'].dt.year
df['is_weekend'] = df['day_of_week'].isin([5, 6])

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
TIME_SLOTS = {
    'night_late': {'start': '00:00', 'end': '06:00', 'id': 0},
    'early_morning': {'start': '06:00', 'end': '09:00', 'id': 1},
    'work_morning': {'start': '09:00', 'end': '12:00', 'id': 2},
    'lunch_break': {'start': '12:00', 'end': '15:00', 'id': 3},
    'work_afternoon': {'start': '15:00', 'end': '18:00', 'id': 4},
    'evening_prime': {'start': '18:00', 'end': '21:00', 'id': 5},
    'night_wind': {'start': '21:00', 'end': '23:59', 'id': 6}
}

def assign_time_slot(timestamp):
    if isinstance(timestamp, str):
        timestamp = pd.to_datetime(timestamp)
    ts_time = timestamp.time()

    for slot_name, slot_info in TIME_SLOTS.items():
        start_time = datetime.strptime(slot_info['start'], '%H:%M').time()
        end_time = datetime.strptime(slot_info['end'], '%H:%M').time()

        if slot_info['id'] < 6:
            if start_time <= ts_time < end_time:
                return slot_info['id']
        else:
            if start_time <= ts_time <= end_time:
                return slot_info['id']

    return -1

df["time_slot_id"] = df["timestamp"].apply(assign_time_slot)

In [ ]:
df.head()

In [ ]:
metrics_columns = ['reactions', 'comments', 'shares']
for col in metrics_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if "post_clicks_by_type" in df.columns:
    df["post_clicks_by_type"] = df["post_clicks_by_type"].ffill() 

In [ ]:
df['total_engagement'] = df['reactions'] + df['comments'] + df['shares']

In [ ]:
df.head()

In [ ]:
print("\n📊 ANALYSE DES PATTERNS DE PUBLICATION PAR ENTREPRISE...")
for ent in df['entreprise'].unique():
    sub = df[df['entreprise'] == ent]
    total_days = (sub['timestamp'].max() - sub['timestamp'].min()).days + 1
    print(f"\n🏢 {ent}:")
    print(f"  - Total posts: {len(sub)}")
    print(f"  - Période: {sub['timestamp'].min().date()} à {sub['timestamp'].max().date()}")
    print(f"  - Posts/jour moyen: {len(sub) / total_days:.2f}")
    print(f"  - Engagement moyen: {sub['total_engagement'].mean():.1f}")
    print(f"  - Engagement médian: {sub['total_engagement'].median():.1f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
fig.suptitle('Analyse Comparative des Patterns de Publication', fontsize=16, fontweight='bold')

entreprises = df['entreprise'].unique()
colors = {'Ooredoo': '#e74c3c', 'Orange': '#f39c12', 'TT': '#3498db'}

# 1. Posts par heure de la journée (comparaison)
for ent in entreprises:
    sub = df[df['entreprise'] == ent]
    hourly_counts = sub.groupby('hour').size()
    hourly_pct = hourly_counts / hourly_counts.sum() * 100  # en %, pour comparer équitablement malgré volumes différents
    axes[0, 0].plot(hourly_pct.index, hourly_pct.values, marker='o', label=ent, color=colors.get(ent))
axes[0, 0].set_title('Distribution par Heure (0-23h)')
axes[0, 0].set_xlabel('Heure de la Journée')
axes[0, 0].set_ylabel('% des Posts')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Posts par jour de la semaine (comparaison)
day_map = {0: 'Lundi', 1: 'Mardi', 2: 'Mercredi', 3: 'Jeudi', 4: 'Vendredi', 5: 'Samedi', 6: 'Dimanche'}
for ent in entreprises:
    sub = df[df['entreprise'] == ent]
    daily_counts = sub.groupby('day_of_week').size()
    daily_pct = daily_counts / daily_counts.sum() * 100
    axes[0, 1].plot([day_map[d] for d in daily_pct.index], daily_pct.values, marker='o', label=ent, color=colors.get(ent))
axes[0, 1].set_title('Distribution par Jour de la Semaine')
axes[0, 1].set_xlabel('Jour')
axes[0, 1].set_ylabel('% des Posts')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Posts par type de contenu (comparaison, barres groupées)
content_pivot = df.groupby(['content_categorie', 'entreprise']).size().unstack(fill_value=0)
content_pivot_pct = content_pivot.div(content_pivot.sum(axis=0), axis=1) * 100
content_pivot_pct.plot(kind='bar', ax=axes[0, 2], color=[colors.get(c) for c in content_pivot_pct.columns])
axes[0, 2].set_title('Distribution par Type de Contenu')
axes[0, 2].set_xlabel('Type de Contenu')
axes[0, 2].set_ylabel('% des Posts')
axes[0, 2].legend(title='Entreprise')
axes[0, 2].grid(True, alpha=0.3)
plt.setp(axes[0, 2].get_xticklabels(), rotation=45, ha='right')

# 4. Évolution temporelle (posts par semaine, une ligne par entreprise)
df['week'] = df['timestamp'].dt.to_period('W')
for ent in entreprises:
    sub = df[df['entreprise'] == ent]
    weekly_counts = sub.groupby('week').size()
    axes[1, 0].plot(weekly_counts.index.to_timestamp(), weekly_counts.values, label=ent, color=colors.get(ent), alpha=0.8)
axes[1, 0].set_title('Évolution Temporelle (Posts/Semaine)')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Posts par Semaine')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 5. Engagement moyen par créneau horaire (comparaison)
for ent in entreprises:
    sub = df[df['entreprise'] == ent]
    slot_eng = sub.groupby('time_slot_id')['total_engagement'].mean()
    axes[1, 1].plot(slot_eng.index, slot_eng.values, marker='o', label=ent, color=colors.get(ent))
axes[1, 1].set_title('Engagement Moyen par Créneau Horaire')
axes[1, 1].set_xlabel('Créneau (0=nuit...6=soir)')
axes[1, 1].set_ylabel('Engagement Moyen')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 6. Engagement moyen par jour de semaine (comparaison)
for ent in entreprises:
    sub = df[df['entreprise'] == ent]
    day_eng = sub.groupby('day_of_week')['total_engagement'].mean()
    axes[1, 2].plot([day_map[d] for d in day_eng.index], day_eng.values, marker='o', label=ent, color=colors.get(ent))
axes[1, 2].set_title('Engagement Moyen par Jour de Semaine')
axes[1, 2].set_xlabel('Jour')
axes[1, 2].set_ylabel('Engagement Moyen')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/analyse_comparative.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np

print("\n INSIGHTS DÉTAILLÉS PAR ENTREPRISE:")

for ent in entreprises:
    sub = df[df['entreprise'] == ent].copy()
    print(f"\n{'='*50}")
    print(f" {ent}")
    print(f"{'='*50}")

    # Heures les plus actives
    hourly_counts = sub.groupby('hour').size()
    top_hours = hourly_counts.sort_values(ascending=False).head(3)
    print(f"\n Heures les plus actives:")
    for hour, count in top_hours.items():
        print(f"  - {hour}h: {count} posts ({count/len(sub)*100:.1f}%)")

    # Jours les plus actifs
    daily_counts = sub.groupby('day_of_week').size()
    print(f"\n Jours les plus actifs:")
    for day, count in daily_counts.sort_values(ascending=False).head(3).items():
        print(f"  - {day_map[day]}: {count} posts ({count/len(sub)*100:.1f}%)")

    # Weekend vs semaine
    weekend_posts = sub[sub['is_weekend']].shape[0]
    weekday_posts = sub[~sub['is_weekend']].shape[0]
    print(f"\n Weekend vs Semaine:")
    print(f"  - Weekend: {weekend_posts} posts ({weekend_posts/len(sub)*100:.1f}%)")
    print(f"  - Semaine: {weekday_posts} posts ({weekday_posts/len(sub)*100:.1f}%)")

    # Heures de bureau vs hors heures
    sub['is_business_hours'] = np.where((sub["hour"] > 8) & (sub["hour"] < 18), True, False)
    business_posts = sub[sub['is_business_hours']].shape[0]
    after_hours_posts = sub[~sub['is_business_hours']].shape[0]
    print(f"\n Heures de bureau (8h-18h) vs Hors heures:")
    print(f"  - Heures de bureau: {business_posts} posts ({business_posts/len(sub)*100:.1f}%)")
    print(f"  - Hors heures: {after_hours_posts} posts ({after_hours_posts/len(sub)*100:.1f}%)")

    # Performance par créneaux
    business_eng = sub[sub['is_business_hours']]['total_engagement'].mean()
    after_eng = sub[~sub['is_business_hours']]['total_engagement'].mean()
    weekend_eng = sub[sub['is_weekend']]['total_engagement'].mean()
    weekday_eng = sub[~sub['is_weekend']]['total_engagement'].mean()
    print(f"\n Performance par créneaux:")
    print(f"  - Engagement heures bureau: {business_eng:.1f}")
    print(f"  - Engagement hors heures: {after_eng:.1f}")
    print(f"  - Engagement weekend: {weekend_eng:.1f}")
    print(f"  - Engagement semaine: {weekday_eng:.1f}")

In [ ]:
from datetime import timedelta

date_ranges = {}
for ent in entreprises:
    sub = df[df['entreprise'] == ent]
    start_date = sub['timestamp'].min().date()
    end_date = sub['timestamp'].max().date()
    dr = pd.date_range(start=pd.to_datetime(start_date), end=pd.to_datetime(end_date) + timedelta(days=1), freq='h')
    date_ranges[ent] = dr
    print(f"✓ {ent} — Grille temporelle: {len(dr)} heures ({start_date} à {end_date})")

In [ ]:
def load_and_prepare_data(files_dict):
    """
    files_dict: dict {nom_entreprise: chemin_fichier}
    Retourne le dataframe complet, nettoyé et enrichi.
    """
    dfs = []
    for name, path in files_dict.items():
        tmp = pd.read_csv(path, encoding='latin-1', on_bad_lines='skip', engine='python')
        tmp['entreprise'] = name
        dfs.append(tmp)
    df = pd.concat(dfs, ignore_index=True)

    # Reactions
    reaction_cols = ["likes", "wow", "sad", "haha", "angry", "love", "thankful"]
    for col in reaction_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['reactions'] = df[reaction_cols].sum(axis=1, min_count=1)

    # Features texte avant suppression du message
    df['message_length'] = df['message'].fillna('').apply(len)
    df['has_hashtag'] = df['message'].fillna('').str.contains('#').astype(int)
    df['has_link'] = df['message'].fillna('').str.contains(r'http|www\.').astype(int)
    df['has_emoji'] = df['message'].fillna('').str.contains(
        r'[\U0001F300-\U0001FAFF\U00002600-\U000027BF]', regex=True
    ).astype(int)

    # Suppression colonnes inutiles (reach_organic/reach_paid conservées)
    df.drop(columns=['created_at','updated_at','is_deleted','is_real','url', 'message','story',
        'picture','profile_cover','local_media','data','views_organic','views_paid','likes','wow',
        'sad', 'haha', 'angry', 'love', 'thankful', 'none',
        'comments_ad','views_ad','reach_ad','likes_ad','saved_ad','shares_ad','reactions_new',
        'score_new','tag','type',"id"], axis=1, inplace=True, errors='ignore')

    df.dropna(axis=1, how="all", inplace=True)

    # Suppression colonnes >90% manquant
    missing_pct = df.isnull().mean() * 100
    cols_to_drop = missing_pct[missing_pct > 90].index.tolist()
    df.drop(columns=cols_to_drop, inplace=True)

    df.drop(columns=["social_id"], inplace=True, errors='ignore')

    # Timestamp
    df = df.rename(columns={"creation_time": "timestamp"})
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%d/%m/%Y %H:%M", errors='coerce')
    df = df.dropna(subset=['timestamp']).reset_index(drop=True)

    # Renommage
    df = df.rename(columns={"status_type": "content_categorie", "social_type": "platform"})
    df.drop(columns=["page_id"], inplace=True, errors='ignore')

    # Features temporelles
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['day_name'] = df['timestamp'].dt.day_name()
    df['month'] = df['timestamp'].dt.month
    df['year'] = df['timestamp'].dt.year
    df['is_weekend'] = df['day_of_week'].isin([5, 6])
    df['is_business_hours'] = (df['hour'] > 8) & (df['hour'] < 18)

    # Créneaux horaires
    df["time_slot_id"] = df["timestamp"].apply(assign_time_slot)

    # Métriques finales
    for col in ['reactions', 'comments', 'shares']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    if "post_clicks_by_type" in df.columns:
        df["post_clicks_by_type"] = df["post_clicks_by_type"].ffill()

    df['total_engagement'] = df['reactions'] + df['comments'] + df['shares']

    return df

In [ ]:
files = {
    'Ooredoo': '../data/raw/posts_dataOOFB0.csv',
    'Orange': '../data/raw/posts_dataOrangeFB0.csv',
    'TT': '../data/raw/posts_dataTTFB0.csv'
}
df = load_and_prepare_data(files)
df.to_csv("../data/prepared/resultPreparation.csv", index=False)
print(f"✓ Export terminé : {df.shape}")

In [ ]:
import pandas as pd
df=pd.read_csv("../data/prepared/resultPreparation.csv")
df.info()